In [ ]:
# Importando todas as Bibliotecas necessárias:
import pandas as pd
import unicodedata

In [ ]:
# Escolhendo o padrão de leitura dos arquivos:
def normalizar_cadastro(valor):
    if pd.isna(valor):
        return None
    valor = str(valor).strip().upper()
    valor = " ".join(valor.split())
    valor = unicodedata.normalize("NFKD", valor).encode("ASCII", "ignore").decode("utf-8")
    return valor

In [ ]:
# 1) Carrega os arquivos
df_ronda = pd.read_parquet(r'df_ronda.parquet')
df_func = pd.read_excel(r'.\xlsx\Lista de funcionários.xlsx', usecols=["NOME", "CHAPA", "FUNÇÃO"])

In [ ]:
# 2) Cria chaves normalizadas para cruzamento por cadastro
df_ronda["cadastro_KEY"] = df_ronda["cadastro"].apply(normalizar_cadastro)
df_func["cadastro_KEY"] = df_func["CHAPA"].apply(normalizar_cadastro)

# 3) Remove duplicidades da base de funcionários
df_func = df_func.drop_duplicates(subset="cadastro_KEY", keep="first")

# 4) Faz o merge para trazer a FUNÇÃO
df_final = df_ronda.merge(
    df_func[["cadastro_KEY", "FUNÇÃO"]],
    on="cadastro_KEY",
    how="left"
)

# 5) Remove coluna auxiliar
df_final = df_final.drop(columns=["cadastro_KEY"])

# 6) Exporta para Excel
df_final.to_excel("SPAC903_82_Final.xlsx", index=False)

print("Arquivo gerado com sucesso: SPAC903_82_Final.xlsx")
print("Linhas com FUNÇÃO preenchida:", df_final["FUNÇÃO"].notna().sum())
print("Linhas sem correspondência:", df_final["FUNÇÃO"].isna().sum())

In [ ]:
# Mostrando análises gerais:
print("Total linhas df_final:", len(df_final))
print("Linhas com FUNÇÃO preenchida:", df_final["FUNÇÃO"].notna().sum())
print("Linhas sem correspondência:", df_final["FUNÇÃO"].isna().sum())

In [ ]:
# Mostrando pessoas sem correspondência:
nao_correspondidas = df_final[df_final["FUNÇÃO"].isna()].copy()
print(
    nao_correspondidas["nome"]
    .dropna()
    .drop_duplicates()
    .sort_values()
    .to_list()
)